In [0]:
CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.dimcustomer
(
    CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID INT,
    CustomerName STRING,
    Email STRING,
    City STRING,
    Address STRING,
    StartDate DATE,
    EndDate DATE,
    IsActive INT
)
USING DELTA;
























































In [0]:
CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.DimProduct
(
    ProductSK BIGINT GENERATED ALWAYS AS IDENTITY,

    ProductID INT,
    ProductName STRING,
    Category STRING,
    UnitPrice DECIMAL(10,2),

    EffectiveDate DATE
);

In [0]:
CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.DimStore
(
    StoreSK BIGINT GENERATED ALWAYS AS IDENTITY,

    StoreID INT,
    StoreName STRING,
    Region STRING
);

In [0]:
CREATE TABLE IF NOT EXISTS gold_catalog.retail_gold.FactSales
(
    SalesSK BIGINT GENERATED ALWAYS AS IDENTITY,

    TransactionID INT,

    CustomerSK BIGINT,
    ProductSK BIGINT,
    StoreSK BIGINT,

    Quantity INT,
    Amount DECIMAL(10,2),

    TxnDate DATE
);

In [0]:
MERGE INTO gold_catalog.retail_gold.DimProduct tgt
USING silver_catalog.retail_silver.silver_products src
ON tgt.ProductID = src.ProductID
WHEN NOT MATCHED THEN
INSERT
(
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    EffectiveDate
)
VALUES
(
    src.ProductID,
    src.ProductName,
    src.Category,
    src.UnitPrice,
    CURRENT_DATE()
);

In [0]:
MERGE INTO gold_catalog.retail_gold.DimStore tgt
USING silver_catalog.retail_silver.silver_stores src
ON tgt.StoreID = src.StoreID
WHEN NOT MATCHED THEN
INSERT
(
    StoreID,
    StoreName,
    Region
)
VALUES
(
    src.StoreID,
    src.StoreName,
    src.Region
);

In [0]:
-- INITIAL LOAD: Run this ONLY ONCE on Day 1 when DimCustomer is empty
-- Comment this out or skip after Day 1

INSERT INTO gold_catalog.retail_gold.DimCustomer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    CURRENT_DATE(),
    DATE('9999-12-31'),
    1
FROM silver_catalog.retail_silver.silver_customers;

In [0]:
--expire old record
MERGE INTO gold_catalog.retail_gold.DimCustomer tgt
USING silver_catalog.retail_silver.silver_customers src

ON tgt.CustomerID = src.CustomerID
AND tgt.IsActive = 1

WHEN MATCHED
AND
(
    tgt.City <> src.City
    OR tgt.Address <> src.Address
)

THEN UPDATE SET
    tgt.IsActive = 0,
    tgt.EndDate = CURRENT_DATE();
     

In [0]:
INSERT INTO gold_catalog.retail_gold.DimCustomer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)

SELECT

    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,

    CURRENT_DATE(),
    DATE('9999-12-31'),
    1

FROM silver_catalog.retail_silver.silver_customers src

WHERE NOT EXISTS
(
    SELECT 1

    FROM gold_catalog.retail_gold.DimCustomer tgt

    WHERE tgt.CustomerID = src.CustomerID
    AND tgt.City = src.City
    AND tgt.Address = src.Address
    AND tgt.IsActive = 1
);

In [0]:
MERGE INTO gold_catalog.retail_gold.FactSales tgt

USING
(
    SELECT

        s.TransactionID,

        c.CustomerSK,
        p.ProductSK,
        st.StoreSK,

        s.Quantity,

        s.Quantity * p.UnitPrice AS Amount,

        s.TxnDate

    FROM silver_catalog.retail_silver.silver_sales s

    JOIN gold_catalog.retail_gold.DimCustomer c
    ON s.CustomerID = c.CustomerID
    AND c.IsActive = 1

    JOIN gold_catalog.retail_gold.DimProduct p
    ON s.ProductID = p.ProductID

    JOIN gold_catalog.retail_gold.DimStore st
    ON s.StoreID = st.StoreID

) src

ON tgt.TransactionID = src.TransactionID

WHEN NOT MATCHED THEN

INSERT
(
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)

VALUES
(
    src.TransactionID,
    src.CustomerSK,
    src.ProductSK,
    src.StoreSK,
    src.Quantity,
    src.Amount,
    src.TxnDate
);

In [0]:
%python
gold_output = "s3a://generalretailsworkspace/processed/"

spark.table("gold_catalog.retail_gold.DimCustomer") \
    .write \
    .mode("overwrite") \
    .option("header",True) \
    .csv(gold_output + "DimCustomer")

spark.table("gold_catalog.retail_gold.DimProduct") \
    .write \
    .mode("overwrite") \
    .option("header",True) \
    .csv(gold_output + "DimProduct")

spark.table("gold_catalog.retail_gold.DimStore") \
    .write \
    .mode("overwrite") \
    .option("header",True) \
    .csv(gold_output + "DimStore")

spark.table("gold_catalog.retail_gold.FactSales") \
    .write \
    .mode("overwrite") \
    .option("header",True) \
    .csv(gold_output + "FactSales")